<a href="https://colab.research.google.com/github/psept77-star/CC_success_probability/blob/main/CC_success_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# 2. Load data
df = pd.read_csv('/content/sample_data/change_control_timeline_1000.csv')

In [ ]:
# Calculate implementation success
df['Implementation_success'] = np.where(df['Linked_Deviations_Count'] == 0, 1, 0)

# Convert dates to datetime object with explicit dayfirst=True to avoid warnings
df['change_initiation_date'] = pd.to_datetime(df['change_initiation_date'], dayfirst=True)
df['change_implementation_date'] = pd.to_datetime(df['change_implementation_date'], dayfirst=True)
df['change_close_date'] = pd.to_datetime(df['change_close_date'], dayfirst=True)

# Calculate days to implement and days to close
df['Days_to_implement'] = (df['change_implementation_date'] - df['change_initiation_date']).dt.days
df['Days_to_close'] = (df['change_close_date'] - df['change_implementation_date']).dt.days

In [ ]:
# Feature selection
num_features = ['Days_to_implement', 'Days_to_close']

# Categorical coulmns
cat_features = ['Department', 'Change_related_to', 'Severity', 'Linked_CAPA_Effectiveness']

# Text columns to vectorize
text_features =  'Change_Description'

# Define target
x = df[num_features + cat_features + [text_features]]
y = df['Implementation_success']

# 3. Split train test data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# 4. Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('text', TfidfVectorizer(max_features=100, stop_words='english'), text_features)
    ]
)

# Model selection
models = {
    'Baseline Logistic Regression': LogisticRegression(max_iter=1000),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

for name, model in models.items():
  clf_pipeline = Pipeline(steps=[('preprocessor',preprocessor),('model', model)])

  clf_pipeline.fit(x_train, y_train)

  y_pred = clf_pipeline.predict(x_test)
  y_prob = clf_pipeline.predict_proba(x_test)[:,1]

  print(f'--------------------{name}--------------------')
  print(classification_report(y_test, y_pred))
  print(f'ROC-AUC score: {roc_auc_score(y_test, y_prob):.4f}')


--------------------Baseline Logistic Regression--------------------
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        81
           1       0.97      1.00      0.98       119

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

ROC-AUC score: 0.9982
--------------------Gradient Boosting--------------------
              precision    recall  f1-score   support

           0       1.00      0.99      0.99        81
           1       0.99      1.00      1.00       119

    accuracy                           0.99       200
   macro avg       1.00      0.99      0.99       200
weighted avg       1.00      0.99      0.99       200

ROC-AUC score: 0.9991


In [ ]:
 #train final production pipeline on full dataset
production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42))
 ])
production_pipeline.fit(x,y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Days_to_implement',
                                                   'Days_to_close']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Department',
                                                   'Change_related_to',
                                                   'Severity',
                                                   'Linked_CAPA_Effectiveness']),
                                                 ('text',
                                                  TfidfVectorizer(max_features=100,
                                                                  stop_words='english'),
                                                  'Change_Description')])),
                ('model', GradientBoostingClassifier(random_state=42))])

In [ ]:
def predict_cc_success_probability(cc_details, pipeline):
  input_df = pd.DataFrame([cc_details])

  init_date = pd.to_datetime(input_df['change_initiation_date'], dayfirst=True)
  impl_date = pd.to_datetime(input_df['change_implementation_date'], dayfirst=True)
  close_date = pd.to_datetime(input_df['change_close_date'], dayfirst=True)

  input_df['Days_to_implement'] = (impl_date - init_date).dt.days
  input_df['Days_to_close'] = (close_date - impl_date).dt.days

  processed_input = input_df[[
      'Days_to_implement', 'Days_to_close', 'Department', 'Change_related_to',
      'Severity', 'Linked_CAPA_Effectiveness', 'Change_Description'
  ]]

  prob = pipeline.predict_proba(processed_input)[0,1]
  return prob


In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create interactive widgets for input
initiation_picker = widgets.DatePicker(description='Initiation Date', value=pd.to_datetime('2025-01-14'))
implementation_picker = widgets.DatePicker(description='Impl. Date', value=pd.to_datetime('2025-02-28'))
close_picker = widgets.DatePicker(description='Close Date', value=pd.to_datetime('2025-04-01'))

department_dropdown = widgets.Dropdown(
    options=["Engineering & Maintenance", "Information Technology", "Manufacturing", "Quality Assurance", "Quality Control", "Regulatory Affairs", "Supply Chain"],
    value="Quality Assurance",
    description='Department'
)

related_to_dropdown = widgets.Dropdown(
    options=["Analytical Methods", "Documentation & SOPs", "Facilities & Utilities", "IT & Automation Systems", "Manufacturing Equipment", "Packaging & Labeling", "Raw Materials & Suppliers"],
    value="Packaging & Labeling",
    description='Related To'
)

severity_dropdown = widgets.Dropdown(
    options=["Minor", "Major", "Critical"],
    value="Major",
    description='Severity'
)

capa_dropdown = widgets.Dropdown(
    options=["Low", "Medium", "High"],
    value="High",
    description='CAPA Effect.'
)

description_text = widgets.Textarea(
    value="Replacement of label application vacuum pads on high speed vial labelers.",
    description='Description',
    layout=widgets.Layout(width='100%', height='80px')
)

predict_button = widgets.Button(
    description='Predict CC Success',
    button_style='primary',
    tooltip='Click to predict implementation success'
)

output_area = widgets.Output()

def format_widget_date(date_obj):
    if date_obj is None:
        return ""
    return date_obj.strftime('%d-%m-%Y')

def handle_prediction(b):
    with output_area:
        clear_output()

        form_input = {
            'change_initiation_date': format_widget_date(initiation_picker.value),
            'change_implementation_date': format_widget_date(implementation_picker.value),
            'change_close_date': format_widget_date(close_picker.value),
            'Department': department_dropdown.value,
            'Change_related_to': related_to_dropdown.value,
            'Severity': severity_dropdown.value,
            'Linked_CAPA_Effectiveness': capa_dropdown.value,
            'Change_Description': description_text.value
        }

        if not form_input['change_initiation_date'] or not form_input['change_implementation_date'] or not form_input['change_close_date']:
            print("Error: Please ensure all dates are selected.")
            return

        try:
            prob = predict_cc_success_probability(form_input, production_pipeline)
            print(f"Prediction Results:")
            print(f"-------------------")
            print(f"Likelihood of CC implementation success without deviations: {prob:.2%}")
        except Exception as e:
            print(f"Could not calculate prediction: {e}")

predict_button.on_click(handle_prediction)

# Structure layout
input_form = widgets.VBox([
    widgets.HTML(value="<h3>Change Control Success Predictor</h3>"),
    widgets.HBox([initiation_picker, implementation_picker, close_picker]),
    widgets.HBox([department_dropdown, related_to_dropdown]),
    widgets.HBox([severity_dropdown, capa_dropdown]),
    description_text,
    predict_button,
    output_area
])

display(input_form)